In [20]:
%load_ext autoreload
%autoreload 2
from experiment import find_all_datasets, find_all_experiments

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
dses = find_all_datasets("../../datasets/")
imagenet = dses["mnist"]
split = "trainUval"

In [4]:
data_dir = "C:/home/eurovis_data/landscape_data_cross_epoch_better/"
strees_dir = "C:/home/eurovis_data/strees_cross_epoch_better/"

In [33]:
exps = find_all_experiments(dses, data_dir, strees_dir)
exp = exps[0]
for e in exps:
	if e.model == "resnet" and e.dataset.name == "mnist" and e.split == split and e.epoch == 193:
		exp = e
		break

exp

resnet (mnist-trainUval), k=20, layer=1, epoch=193

In [6]:
from basic_utils import get_labels, get_partition, get_tree, get_order_and_weights
labels = get_labels(exp)
partition = get_partition(exp)

assert len(partition) == len(labels)

In [7]:
import pyct as ct

data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

homo_prop = 0.99
fns, counts = simpl.getHomoValleyPlot(order, wts, labels, homo_prop, partition) # type: ignore
fns_norm, counts_norm_min, counts_norm_max = simpl.getSimplificationPlot(order, wts)

In [8]:
import plotly.express as px

px.line(x=fns, y=counts, labels={"x": "Function Value", "y": "Number of Homogeneous Valleys"}, title=f"Homogeneous Valleys vs Function Value (Homogeneity Threshold = {homo_prop})")

In [9]:
px.line(x=fns_norm, y=counts_norm_min, labels={"x": "Function Value", "y": "Number of Valleys"}, title="Number of Valleys vs Function Value")

In [10]:
data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

fns_more, remaining_all, remaining_homo, maj_class_homo_cov, maj_class_homo_counts, class_homo_covs, class_all_covs = simpl.getHomoValleyPlotPlusCoverages(order, wts, labels, homo_prop, partition)

In [11]:
list(zip(maj_class_homo_cov, class_all_covs))

[([0.008836737650296972,
   0.015107274343023993,
   0.007868383404864092,
   0.007141856882789525,
   0.006887456037514654,
   0.00887058450815777,
   0.006253635834787667,
   0.00658165364047717,
   0.00673992673992674,
   0.005748778384593274],
  [0.008836737650296972,
   0.015107274343023993,
   0.007868383404864092,
   0.007141856882789525,
   0.006887456037514654,
   0.00887058450815777,
   0.006253635834787667,
   0.00658165364047717,
   0.00673992673992674,
   0.005748778384593274]),
 ([0.008691873098652759,
   0.015107274343023993,
   0.007868383404864092,
   0.007141856882789525,
   0.006887456037514654,
   0.00887058450815777,
   0.006253635834787667,
   0.00658165364047717,
   0.00673992673992674,
   0.005748778384593274],
  [0.008691873098652759,
   0.015107274343023993,
   0.007868383404864092,
   0.007141856882789525,
   0.006887456037514654,
   0.00887058450815777,
   0.006253635834787667,
   0.00658165364047717,
   0.00673992673992674,
   0.005748778384593274]),
 ([0.0

In [25]:
idx = remaining_all.index(10)
print(idx, fns[idx], maj_class_homo_cov[idx], class_all_covs[idx], maj_class_homo_counts[idx], sep="\n")

370
1.2993490599910729e-05
[0.946834709546574, 0.9645804240192967, 0.9081545064377683, 0.907716006161602, 0.9638042203985931, 0.8810391256138128, 0.9472076788830716, 0.9310297545591664, 0.9201465201465201, 0.9406438631790744]
[0.946834709546574, 0.9647073759045321, 0.9082975679542203, 0.907716006161602, 0.9638042203985931, 0.8811975289086013, 0.9472076788830716, 0.9310297545591664, 0.9201465201465201, 0.9406438631790744]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [32]:
classes_needed = 10 # 10 classes, the following criteria must apply to this many of them
maj_coverage_needed = 0.5 # at least 50% coverage in homogeneous valleys where they are the only class (100% proportion in valley) 
total_coverage_needed = 0.8 # at least 80% total coverage in all valleys

# coverages are functions of accuracy, so this constraint necessarily tightens in more complex datasets

represented = [[mc >= maj_coverage_needed and tc >= total_coverage_needed for mc, tc in zip(maj_cov, total_cov)].count(True) >= classes_needed 
            	for maj_cov, total_cov in zip(maj_class_homo_cov, class_all_covs)]

# find interval where this is true
last_idx = len(fns) - represented[::-1].index(True) - 1

first_idx = represented.index(True)

print(f"Uniform Interval: {all(represented[first_idx:last_idx+1])}")
print(f"First IDX: {first_idx} - ")
print(first_idx, len(fns), fns[first_idx], list(zip(maj_class_homo_cov[first_idx], class_all_covs[first_idx], maj_class_homo_counts[first_idx])), sep="\n")
print(f"\nLast IDX: {last_idx} - ")
print(last_idx, len(fns), fns[last_idx], list(zip(maj_class_homo_cov[last_idx], class_all_covs[last_idx], maj_class_homo_counts[last_idx])), sep="\n")

Uniform Interval: True
First IDX: 370 - 
370
380
1.2993490599910729e-05
[(0.946834709546574, 0.946834709546574, 1), (0.9645804240192967, 0.9647073759045321, 1), (0.9081545064377683, 0.9082975679542203, 1), (0.907716006161602, 0.907716006161602, 1), (0.9638042203985931, 0.9638042203985931, 1), (0.8810391256138128, 0.8811975289086013, 1), (0.9472076788830716, 0.9472076788830716, 1), (0.9310297545591664, 0.9310297545591664, 1), (0.9201465201465201, 0.9201465201465201, 1), (0.9406438631790744, 0.9406438631790744, 1)]

Last IDX: 370 - 
370
380
1.2993490599910729e-05
[(0.946834709546574, 0.946834709546574, 1), (0.9645804240192967, 0.9647073759045321, 1), (0.9081545064377683, 0.9082975679542203, 1), (0.907716006161602, 0.907716006161602, 1), (0.9638042203985931, 0.9638042203985931, 1), (0.8810391256138128, 0.8811975289086013, 1), (0.9472076788830716, 0.9472076788830716, 1), (0.9310297545591664, 0.9310297545591664, 1), (0.9201465201465201, 0.9201465201465201, 1), (0.9406438631790744, 0.9406438